In [1]:
"""
Этап 11 (бонус). Basket analysis.

Ищем товары, которые часто покупают вместе.
Используем упрощённую версию: топ-100 SKU по выручке.

Метрики:
- Support: доля заказов с товаром.
- Confidence: P(B|A).
- Lift: P(A,B) / (P(A)*P(B)).
"""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from scripts.db_config import read_sql, execute_sql, _get_conn_params

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

REPORTS_DIR = ROOT_DIR / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

print("✅ Окружение готово")

✅ Окружение готово


In [2]:
# Сколько SKU в разных регистрах?
check = read_sql("""
    SELECT
        stock_code,
        UPPER(stock_code) AS normalized,
        COUNT(*) AS n
    FROM raw_transactions
    WHERE stock_code != UPPER(stock_code)
    GROUP BY stock_code, UPPER(stock_code)
    ORDER BY n DESC
    LIMIT 20;
""")
print(check)

   stock_code normalized    n
0      18098c     18098C  142
1      15056n     15056N  121
2      72349b     72349B  121
3      47566b     47566B  116
4      84970s     84970S  103
5      85123a     85123A  101
6     15056bl    15056BL   93
7      84997b     84997B   90
8      82494l     82494L   88
9      84997d     84997D   86
10     84997a     84997A   81
11     84997c     84997C   80
12     84509a     84509A   75
13     47591d     47591D   74
14     84970l     84970L   71
15     85049e     85049E   64
16     72801d     72801D   61
17     84510a     84510A   59
18     84873a     84873A   59
19     85049a     85049A   53


In [3]:
check = read_sql("""
    SELECT
        stock_code,
        COUNT(*) AS n,
        SUM(quantity * unit_price) AS revenue
    FROM raw_transactions
    WHERE UPPER(stock_code) = '85099F'
    GROUP BY stock_code;
""")
print(check)

  stock_code     n   revenue
0     85099F  1916  67976.57
1     85099f    27   1077.08


In [4]:
# Сколько SKU в разных регистрах?
check = read_sql("""
    SELECT
        stock_code,
        UPPER(stock_code) AS normalized,
        COUNT(*) AS n,
        SUM(quantity * unit_price) AS revenue
    FROM raw_transactions
    WHERE stock_code != UPPER(stock_code)
    GROUP BY stock_code, UPPER(stock_code)
    ORDER BY revenue DESC
    LIMIT 30;
""")
print(f"Всего проблемных SKU: {len(check)}")
print(f"Их общая выручка: £{check['revenue'].sum():,.2f}")
display(check)

Всего проблемных SKU: 30
Их общая выручка: £45,983.08


,stock_code,normalized,n,revenue
0,84997b,84997B,90,4534.32
1,85123a,85123A,101,3992.39
2,82494l,82494L,88,3545.55
3,84997d,84997D,86,3258.31
4,84997c,84997C,80,2984.28
5,15056n,15056N,121,2259.44
6,85099b,85099B,25,2111.64
7,47566b,47566B,116,1803.61
8,18098c,18098C,142,1682.01
9,85049a,85049A,53,1669.64


In [5]:
from scripts.db_config import read_sql

# Проверка 1: есть ли ещё строчные буквы?
check = read_sql("""
    SELECT COUNT(DISTINCT stock_code) AS problematic
    FROM raw_transactions
    WHERE stock_code != UPPER(stock_code);
""")
print(f"SKU с неправильным регистром: {check['problematic'].iloc[0]}")

# Проверка 2: 85099f vs 85099F
check = read_sql("""
    SELECT
        stock_code,
        COUNT(*) AS n,
        SUM(quantity * unit_price) AS revenue
    FROM raw_transactions
    WHERE stock_code LIKE '85099%'
    GROUP BY stock_code
    ORDER BY revenue DESC;
""")
display(check)

SKU с неправильным регистром: 0


,stock_code,n,revenue
0,85099B,4241,183390.15
1,85099F,1943,69053.65
2,85099C,1962,59806.61


In [6]:
check = read_sql("""
    SELECT 
        stock_code,
        COUNT(*) AS n
    FROM raw_transactions
    WHERE stock_code IS NULL OR stock_code = ''
    GROUP BY stock_code
    LIMIT 10;
""")
display(check)

# И посмотрим примеры строк с пустым stock_code
check2 = read_sql("""
    SELECT *
    FROM raw_transactions
    WHERE stock_code IS NULL OR stock_code = ''
    LIMIT 20;
""")
display(check2)

,stock_code,n
0,None,932385


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,491041,None,6 HOOK JEWELLERY STAND BLACK DRESS,7,2009-12-09 09:42:00,8.65,None,United Kingdom
1,491041,None,TOP SECRET PEN SET,2,2009-12-09 09:42:00,2.57,None,United Kingdom
2,491041,None,BUTTONS DESIGN JEWELLERY BOX,1,2009-12-09 09:42:00,7.78,None,United Kingdom
3,491041,None,6 SEGMENT COLOURED SNACK BOWL,1,2009-12-09 09:42:00,16.48,None,United Kingdom
4,491041,None,GIRLS ALPHABET IRON ON PATCHES,1,2009-12-09 09:42:00,0.43,None,United Kingdom
5,491041,None,ELEPHANT TOY WITH BLUE T-SHIRT,1,2009-12-09 09:42:00,4.3,None,United Kingdom
6,491041,None,MOUSE TOY WITH PINK T-SHIRT,2,2009-12-09 09:42:00,4.3,None,United Kingdom
7,491041,None,PINK GINGHAM CAT WITH SCARF,1,2009-12-09 09:42:00,5.17,None,United Kingdom
8,491041,None,BOYS ALPHABET IRON ON PATCHES,3,2009-12-09 09:42:00,0.43,None,United Kingdom
9,491041,None,BOX OF 24 COCKTAIL PARASOLS,1,2009-12-09 09:42:00,0.83,None,United Kingdom


In [7]:
check = read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE stock_code IS NULL OR stock_code = '') AS null_stock,
        COUNT(DISTINCT stock_code) AS unique_stocks,
        MIN(invoice_date) AS min_date,
        MAX(invoice_date) AS max_date
    FROM raw_transactions;
""")
print(check.T)

                                 0
total_rows                 1067371
null_stock                       0
unique_stocks                 5131
min_date       2009-12-01 07:45:00
max_date       2011-12-09 12:50:00


In [10]:
# Проверка топ-10 SKU
check = read_sql("""
    SELECT
        stock_code,
        description,
        COUNT(DISTINCT invoice_no) AS orders,
        SUM(quantity) AS units,
        ROUND(SUM(revenue)::numeric, 2) AS revenue,
        ROUND(SUM(revenue)::numeric / (SELECT SUM(revenue) FROM sales) * 100, 2) AS share_pct
    FROM sales
    GROUP BY stock_code, description
    ORDER BY revenue DESC
    LIMIT 10;
""")
display(check)

,stock_code,description,orders,units,revenue,share_pct
0,22423,REGENCY CAKESTAND 3 TIER,3918,27577,344563.25,1.71
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,5455,96683,266923.55,1.33
2,23843,"PAPER CRAFT , LITTLE BIRDIE",1,80995,168469.60,0.84
3,85099B,JUMBO BAG RED RETROSPOT,3269,79279,150935.56,0.75
4,47566,PARTY BUNTING,2674,28378,149187.05,0.74
5,84879,ASSORTED COLOUR BIRD ORNAMENT,2807,81809,132187.92,0.66
6,22086,PAPER CHAIN KIT 50'S CHRISTMAS,2018,36581,123141.54,0.61
7,79321,CHILLI LIGHTS,1135,16840,85489.91,0.43
8,23166,MEDIUM CERAMIC TOP STORAGE JAR,247,78033,81700.92,0.41
9,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,756,32525,74448.92,0.37


In [13]:
check = read_sql("""
    SELECT abc_category, COUNT(*) AS n, ROUND(SUM(revenue)::numeric, 2) AS revenue
    FROM abc
    GROUP BY abc_category
    ORDER BY abc_category;
""")
print(check)

  abc_category     n      revenue
0            A  1024  16084297.24
1            B  1260   3019022.16
2            C  2439   1005699.27


In [15]:
# Проверка ключевых SKU
check = read_sql("""
    SELECT purchase_invoice, cancel_invoice, stock_code, quantity, customer_id
    FROM cancelled_pairs
    WHERE stock_code IN ('23843', '23166')
    ORDER BY quantity DESC;
""")
print("Пары по ключевым SKU:")
display(check)

Пары по ключевым SKU:


,purchase_invoice,cancel_invoice,stock_code,quantity,customer_id
0,581483,C581484,23843,80995,16446
1,541431,C541433,23166,74215,12346


In [16]:
# Диагностика: сколько выручки потеряли из-за каждой причины?
check1 = read_sql("""
    SELECT ROUND(SUM(cp.quantity * rt.unit_price)::numeric, 2) AS cancelled_pairs_revenue
    FROM cancelled_pairs cp
    INNER JOIN raw_transactions rt
        ON rt.invoice_no = cp.purchase_invoice
        AND rt.stock_code = cp.stock_code
        AND rt.quantity = cp.quantity;
""")
print(f"Выручка исключённых пар: £{check1.iloc[0, 0]:,.2f}")

check2 = read_sql("""
    SELECT ROUND(SUM(quantity * unit_price)::numeric, 2) AS dec2011_revenue
    FROM raw_transactions
    WHERE invoice_no NOT LIKE 'C%'
      AND quantity > 0
      AND unit_price > 0
      AND invoice_date >= '2011-12-01'
      AND invoice_date <= '2011-12-09';
""")
print(f"Выручка 1-9 декабря 2011: £{check2.iloc[0, 0]:,.2f}")

Выручка исключённых пар: £284,219.23
Выручка 1-9 декабря 2011: £437,872.08


In [1]:
check = read_sql("""
    SELECT
        COUNT(DISTINCT stock_code) AS unique_skus,
        COUNT(DISTINCT invoice_no) AS unique_invoices,
        COUNT(DISTINCT customer_id) AS unique_customers,
        SUM(quantity) AS total_units,
        ROUND(SUM(revenue)::numeric, 2) AS total_revenue,
        MIN(invoice_date) AS min_date,
        MAX(invoice_date) AS max_date
    FROM sales;
""")
print(check.T)

NameError: name 'read_sql' is not defined